In [0]:
files = dbutils.fs.ls("/Volumes/spotify-data-project-dev/default/spotify-data-project")
display(files)

path,name,size,modificationTime
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_albums.csv,spotify_albums.csv,2631253,1787223670000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_artist_features.csv,spotify_artist_features.csv,12015713,1787223675000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_artists.csv,spotify_artists.csv,8264572,1787223673000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_audio_features.csv,spotify_audio_features.csv,270402495,1787223708000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_decade_trends.csv,spotify_decade_trends.csv,1137,1787223668000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_features.csv,spotify_features.csv,456178368,1787223708000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_genre_tags.csv,spotify_genre_tags.csv,142851623,1787223708000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_playlist_cooccurrence.csv,spotify_playlist_cooccurrence.csv,191578962,1787223708000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_popularity_metrics.csv,spotify_popularity_metrics.csv,175451915,1787223708000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/taste_cluster_report.csv,taste_cluster_report.csv,23446,1787223668000


In [0]:
import pyspark.sql.functions as F

df_raw = spark.read.csv("/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_artists.csv", header=True)


display(df_raw.limit(10))
display(df_raw.columns)

artist_id,name,name_normalized,country_of_origin,career_start_year,primary_genre_l1,primary_genre_l2,spotify_followers,spotify_popularity,monthly_listeners,musicbrainz_id,lastfm_listeners,lastfm_play_count,is_solo_artist,source_dataset
artist:000347dd-e0b0-a87a-8638-ea93e2fa1653,Baskerville,baskerville,null,2017,Metal,Heavy Metal,null,null,null,null,null,null,null,compiled_multi_source
artist:00048daf-510a-51ec-64c9-a347cc29b997,Shion Tsuji,shion tsuji,null,2009,Folk,Singer-Songwriter,null,null,null,null,null,null,null,compiled_multi_source
artist:00057abd-bcac-477a-074a-7e93e217674e,Yu Takahashi,yu takahashi,null,2010,Folk,Acoustic,null,null,null,null,null,null,null,compiled_multi_source
artist:0006978c-83ee-4981-1dd4-764bc9b27ef4,Jimmy Gourley,jimmy gourley,null,2004,Instrumental,Guitar,null,null,null,null,null,null,null,compiled_multi_source
artist:00080783-707a-2414-a10e-00aa1c54622a,J Bas Y Santy,j bas y santy,null,2019,Latin,Latin,null,null,null,null,null,null,null,compiled_multi_source
artist:0008fec2-3531-0f7c-12b9-1a7563bdc77e,TK N Cash,tk n cash,null,2014,Hip-Hop,Hip-Hop,null,null,null,null,null,null,null,compiled_multi_source
artist:00090478-00b5-21c8-65c4-c4f308a979c9,Tango Siempre,tango siempre,null,2008,Latin,Tango,null,null,null,null,null,null,null,compiled_multi_source
artist:0009cfc6-a2f4-d888-2beb-aa5530427829,Kideko,kideko,null,2015,Electronic,Deep House,null,null,null,null,null,null,null,compiled_multi_source
artist:000a2f01-80fd-07f9-29d1-9c86e08a6bd1,Chris Young,chris young,null,2006,Country,Country,2918550.0,65.0,null,null,null,null,null,compiled_multi_source
artist:000c69c5-05fa-6523-313e-e2387ee41f4b,Lonely God,lonely god,null,2018,Pop,Indie Pop,null,null,null,null,null,null,null,compiled_multi_source


_1
artist_id
name
name_normalized
country_of_origin
career_start_year
primary_genre_l1
primary_genre_l2
spotify_followers
spotify_popularity
monthly_listeners


In [0]:
try:
    dbutils.fs.rm("/Volumes/spotify-data-project-dev/default/checkpoints/read_v3", True)
    dbutils.fs.rm("/Volumes/spotify-data-project-dev/default/checkpoints/write_artists_v3", True)
    print("Cleared checkpoint directories")
except Exception as e:
    print(f"Could not clear checkpoints: {e}")

Cleared checkpoint directories


In [0]:
# Process bronze artists using batch read and write
def process_bronze_artist(): 
    df = spark.read.csv(
        "/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_artists.csv",
        header=True,
        inferSchema=True
    ) \
        .withColumn("timestamp_added", F.current_timestamp()) \
        .withColumn("year_month", F.date_format("timestamp_added", "yyyy-MM"))
    
    df.write \
        .mode("append") \
        .partitionBy("primary_genre_l1", "primary_genre_l2", "timestamp_added") \
        .saveAsTable("bronze_artists")
    
    print(f"Wrote {df.count()} rows to bronze_artists")

process_bronze_artist()

Wrote 69414 rows to bronze_artists
